In [19]:
%run AgentRegistry.ipynb

Hier sind die günstigsten Flüge von Köln (DUS) nach Tunesien:

* Airline: Lufthansa
* Flugnummer: LH 155
* Abflug- und Ankunftszeit: 09:00 Uhr (Köln) / 14:20 Uhr (Tunis)
* Preis: 240 €

* Airline: Turkish Airlines
* Flugnummer: TK 219
* Abflug- und Ankunftszeit: 10:30 Uhr (Köln) / 17:05 Uhr (Tunis)
* Preis: 260 €

* Airline: Ryanair
* Flugnummer: FR 1234
* Abflug- und Ankunftszeit: 11:15 Uhr (Köln) / 18:30 Uhr (Tunis)
* Preis: 200 €

Bitte beachten Sie, dass die Preise und Flugzeiten je nach Saison und Verfügbarkeit variieren können. Es lohnt sich, die Preise und Flugzeiten vor der Buchung zu überprüfen.
Am 20.11.2026 ist in Berlin folgendes Wettervorhersage:

* Temperatur: 0,4°C (Morgen: 1,5°C)
* Regen: 40%
* Sonne scheint: Nein

Es wird einen geringen Regen geben, aber die Sonne wird nicht scheinen.
{'Flight_Agent': <langgraph.graph.state.CompiledStateGraph object at 0x0000019DA16BE390>, 'Weather_Agent': <langgraph.graph.state.CompiledStateGraph object at 0x0000019D9B964110>}
{'Fligh

In [22]:
from langgraph.graph import StateGraph, START , END
import operator
from typing import TypedDict, Annotated

In [23]:
class State(TypedDict):
    anfrage: str
    ziele : list
    ergebnisse : Annotated[list, operator.add]
    bericht : str

In [24]:
def coordinator(state):
    namen = list(registry.agents)
    prompt = (f"Agenten und Aufgaben: {registry.list_beschreibungen()}\n"
              f"Anfrage: {state['anfrage']}\n"
              f"Wähle alle passenden Agenten aus {namen}"
              "Antworte nur mit den Namen , kommagetrennt , ohne weiteren Text."
              )
    antwort = llm.invoke(prompt).content
    return {"ziele": [n for n in registry.agents if n in antwort]}

In [25]:
def route(state):
    return state["ziele"] or ["report"]

In [26]:
def make_node(name):
    def node(state):
        antwort = registry.call(name,state["anfrage"])
        return {"ergebnisse": [{"von":name,"inhalt":antwort}]}
    return node

In [27]:
def report(state):
    if not state["ergebnisse"]:
        return {"bericht": "Kein Agent konnte die Anfrage beantworten"}
    text = llm.invoke(
        f"Anfrage: {state['anfrage']}"
        f"Ergebnisse: {state['ergebnisse']}\n"
        "Fasse Nur diese Ergebnisse kurz auf Deutsch zusammen. Erfinde nichts."
    ).content
    return {"bericht": text}

In [28]:
g = StateGraph(State)
g.add_node("coordinator", coordinator)
g.add_node("report",report)
g.add_edge(START,"coordinator")

In [29]:
for name in registry.list_agents():
    g.add_node(name, make_node(name))
    g.add_edge(name, "report")

In [30]:
g.add_conditional_edges("coordinator", route)
g.add_edge("report", END)

app = g.compile()

In [34]:
res = app.invoke({"anfrage": "Wetter in Köln und Flüge nach Tunis am 25.11.2026", "ergebnisse": []})
print(res["bericht"])

Hier ist eine kurze Zusammenfassung der Ergebnisse:

*   Flug nach Tunis am 25.11.2026: Eurowings EW1234, Abflugszeit 06:00 Uhr, Ankunftszeit 09:00 Uhr, Preis: 150 €. Bitte beachten Sie, dass sich die Informationen ändern können.
*   Wettervorhersage für Köln am 25.11.2026: Sonnig, Temperatur ca. 2,5°C, geringe Regenwahrscheinlichkeit. Bitte überprüfen Sie vor Ihrer Reise die aktuellsten Informationen.
